<a href="https://colab.research.google.com/github/Devp1p/oad/blob/main/pr6Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

df = pd.read_csv("titanic.csv")
print(df.head())
print("Размер исходных данных:", df.shape)


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
Ра

## 1. Подготовка данных

In [8]:
features = ["Pclass", "Age", "Sex", "Fare", "SibSp", "Parch"]
X = df[features].copy()
y = df["Survived"]

X["Age"] = X["Age"].fillna(X["Age"].median())
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

print("Пропуски в признаках:")
print(X.isnull().sum())
print("\nТипы данных:")
print(X.dtypes)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nРазмер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)
print("Размер y_train:", y_train.shape)
print("Размер y_test:", y_test.shape)


Пропуски в признаках:
Pclass    0
Age       0
Sex       0
Fare      0
SibSp     0
Parch     0
dtype: int64

Типы данных:
Pclass      int64
Age       float64
Sex         int64
Fare      float64
SibSp       int64
Parch       int64
dtype: object

Размер X_train: (712, 6)
Размер X_test: (179, 6)
Размер y_train: (712,)
Размер y_test: (179,)


**Ответ на вопрос:** тестовая выборка нужна для независимой оценки качества на данных, которые модель не использовала при обучении. `scaler.fit_transform()` применяется только к обучающей выборке, чтобы параметры масштабирования не подсматривали в тестовые данные; к тестовой выборке применяется только `transform()`.

## 2. Логистическая регрессия и предсказания

In [9]:
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)
print("Первые предсказания:", y_pred_lr[:10])
print("Правильные ответы:  ", y_test.iloc[:10].to_list())


Первые предсказания: [0 0 0 0 1 0 1 0 0 0]
Правильные ответы:   [0, 0, 1, 0, 1, 1, 1, 0, 0, 0]


**Ответ:** `X_train_scaled` — признаки обучающей выборки, `y_train` — правильные классы для обучения. Тестовые данные не используются при обучении, чтобы оценка была честной. Значение `0` означает, что пассажир не выжил, `1` — что выжил. Сравните две строки выше: совпадения — правильные предсказания, различия — ошибки.

## 3. Метрики логистической регрессии

In [10]:
def show_metrics(name, y_true, y_pred):
    print(f"--- {name} ---")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("Матрица ошибок [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print(classification_report(y_true, y_pred, zero_division=0))

show_metrics("Логистическая регрессия", y_test, y_pred_lr)


--- Логистическая регрессия ---
Accuracy: 0.8045
Precision: 0.7656
Recall: 0.7101
F1: 0.7368
Матрица ошибок [[TN, FP], [FN, TP]]:
[[95 15]
 [20 49]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       110
           1       0.77      0.71      0.74        69

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



**Ответы:** Accuracy — доля правильных предсказаний. Recall для класса `1` показывает, какую долю реально выживших модель обнаружила. Ложноположительный результат (FP) — модель предсказала выживание, хотя пассажир не выжил. Ложноотрицательный (FN) — модель предсказала, что пассажир не выжил, хотя он выжил.

## 4. Дерево решений и сравнение моделей

In [11]:
model_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
model_tree.fit(X_train, y_train)
y_pred_tree = model_tree.predict(X_test)

show_metrics("Дерево решений", y_test, y_pred_tree)

comparison = pd.DataFrame({
    "Модель": ["Логистическая регрессия", "Дерево решений"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_tree)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr, zero_division=0),
        precision_score(y_test, y_pred_tree, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_tree, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_tree, zero_division=0)
    ]
})
comparison.round(4)


--- Дерево решений ---
Accuracy: 0.7821
Precision: 0.875
Recall: 0.5072
F1: 0.6422
Матрица ошибок [[TN, FP], [FN, TP]]:
[[105   5]
 [ 34  35]]
              precision    recall  f1-score   support

           0       0.76      0.95      0.84       110
           1       0.88      0.51      0.64        69

    accuracy                           0.78       179
   macro avg       0.82      0.73      0.74       179
weighted avg       0.80      0.78      0.77       179



,Модель,Accuracy,Precision,Recall,F1
0,Логистическая регрессия,0.8045,0.7656,0.7101,0.7368
1,Дерево решений,0.7821,0.8750,0.5072,0.6422


**Ответ:** обе модели используют одни и те же `X_train`, `X_test`, `y_train` и `y_test`, поэтому сравнение корректное. По таблице выше определите, у какой модели выше Accuracy и F1; не вписывайте числа заранее — они зависят от конкретного файла `titanic.csv`.

## 5. Связь с корреляционным анализом из ПР № 5

In [12]:
corr_df = df[["Survived", "Pclass", "Age", "Fare", "SibSp", "Parch"]].copy()
corr_df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
corr = corr_df.corr(numeric_only=True)["Survived"].drop("Survived").sort_values()
print(corr)
print("\nНаиболее сильная положительная корреляция:", corr.idxmax(), round(corr.max(), 4))
print("Наиболее сильная отрицательная корреляция:", corr.idxmin(), round(corr.min(), 4))


Pclass   -0.338481
Age      -0.077221
SibSp    -0.035322
Parch     0.081629
Fare      0.257307
Sex       0.543351
Name: Survived, dtype: float64

Наиболее сильная положительная корреляция: Sex 0.5434
Наиболее сильная отрицательная корреляция: Pclass -0.3385


## Вывод

В работе подготовлены признаки пассажиров Titanic, данные разделены на обучающую и тестовую выборки, обучены две модели бинарной классификации и рассчитаны Accuracy, Precision, Recall, F1 и матрицы ошибок. Фактические значения метрик и названия признаков с максимальной/минимальной корреляцией нужно взять из вывода ячеек после запуска ноутбука на `titanic.csv`.